# 店家描述类别分类

本 Notebook 以补名草稿中的 `name` 和全部 `ratings.csv` 评论为证据，为店家新增 `描述类别`。分类采用“名称关键词权重 + 评论关键词出现次数”的可解释规则；无法获得足够证据的店家保留为“不确定”。


## 第 1 步：配置输入与输出路径


In [49]:
from pathlib import Path
import csv
import re
from collections import Counter, defaultdict

PROJECT_DIR = Path(r"D:\携程实习文件\项目2\推荐系统数据集\推荐系统数据集\店家")
RAW_DIR = PROJECT_DIR / "data" / "raw"
DERIVED_DIR = PROJECT_DIR / "data" / "derived"
FINAL_DIR = PROJECT_DIR / "data" / "final"
RATING_SPLIT_DIR = PROJECT_DIR / "data" / "splits" / "rating_split"
NAME_OUTPUT_DIR = PROJECT_DIR / "outputs" / "name_completion"
CATEGORY_OUTPUT_DIR = PROJECT_DIR / "outputs" / "category"
BASE_RESTAURANTS_PATH = DERIVED_DIR / "restaurants_name_draft_selected_candidates.csv"
RATINGS_PATH = RAW_DIR / "ratings.csv"
OUTPUT_PATH = DERIVED_DIR / "restaurants_category_draft.csv"
SUMMARY_PATH = CATEGORY_OUTPUT_DIR / "restaurant_category_summary.csv"


## 第 2 步：定义类别与关键词

名称命中关键词的权重为 5；每条评论对同一类别最多贡献 1 分，避免长评论因重复用词被过度放大。


In [50]:
CATEGORY_KEYWORDS = {
    "电商平台/线上零售": ("淘宝", "京东", "当当", "易迅", "唯品会", "凡客", "亚马逊", "1号店", "天猫商城", "卓越网"),
    "超市/便利店": ("超市", "便利店", "大卖场", "沃尔玛", "家乐福", "全家", "7-11"),
    "市场/花鸟市场": ("花鸟市场", "农贸市场", "水产市场", "菜市场", "花市", "花鸟", "水族"),
    "景点/公园/公共场所": ("森林公园", "公园", "广场", "博物馆", "风景区", "景区", "游乐园", "动物园", "植物园", "度假村"),
    "教育培训/驾校": ("驾校", "学车", "教练", "驾照", "培训班", "练车", "幼儿园", "中学", "重点中学"),
    "餐饮-火锅烧烤": ("火锅", "烧烤", "烤串", "串串", "烤肉", "烤鱼", "烤地瓜", "烤翅", "羊蝎子", "麻辣香锅", "泡椒", "汉堡", "菜饭", "骨头汤", "肉粽", "烤羊腿", "自助餐"),
    "餐饮-茶饮咖啡甜品": ("奶茶", "咖啡", "甜品", "蛋糕", "蛋挞", "面包", "冰淇淋", "果汁", "爆米花", "小丸子", "茶馆", "茶坊", "茶饮", "红茶", "锡兰茶", "凉茶", "品茶", "茶叶", "茶叶礼盒"),
    "酒吧/夜生活": ("酒吧", "夜店", "鸡尾酒", "驻唱", "夜场", "啤酒"),
    "酒店/住宿": ("酒店", "宾馆", "客房", "住宿", "住店", "旅舍", "前台", "招待所"),
    "购物零售": ("商场", "服装", "衣服", "鞋", "化妆品", "专卖", "购物", "眼镜", "梨膏糖", "巧克力", "床垫", "书店", "粉扑", "耳钉", "饰品", "银饰"),
    "生活服务": ("美发", "理发", "美容", "SPA", "按摩", "洗浴", "修车", "干洗", "摄影", "牙科", "口腔", "拔牙", "宠物医院", "宠物诊所", "中医诊所", "儿科", "妇科", "中药", "药房", "银行"),
    "休闲娱乐": ("KTV", "影院", "电影", "桌游", "游泳", "健身", "羽毛球", "演出", "卡拉OK", "台球", "乒乓球", "网吧", "录音棚"),
    "餐饮-正餐": (
        "餐厅", "饭店", "饭馆", "餐馆", "小吃", "美食", "菜馆", "食府", "酒家", "酒楼", "食堂", "面馆", "面店",
        "拉面", "米线", "米粉", "螺蛳粉", "麻辣烫", "饺子", "煎饺", "馄饨", "包子", "粥", "炒饭", "盖饭", "盖浇饭", "快餐", "便当", "盒饭",
        "烤鸭", "烤鸡", "烧鸡", "白切鸡", "鸡煲", "土鸡", "烧鹅", "卤鹅", "鹅肝", "鸭脖", "卤味", "卤菜", "肉夹馍", "凉皮", "胡辣汤", "臊子面", "牛杂", "烧饼", "煎饼",
        "锅贴", "生煎", "锅盔", "馅饼", "油条", "油饼", "酸菜鱼", "水煮鱼", "鸡公煲", "乳鸽鸡煲", "煲仔饭", "牛肉面", "牛肉", "牛排", "牛蛙", "羊肉泡馍", "泡馍",
        "烤土鸡", "猪蹄", "海鲜", "乳鸽", "大盘鱼", "水饺", "鲜捞", "凉拌", "小肠", "香干", "罗汉肉泡粉", "中餐", "炖品", "寿司", "三文鱼", "蟹沙拉", "大排档", "白斩鸡饭", "猪扒包", "披萨", "培根", "芝士焗蘑菇", "薯条", "牛肚", "羊肉", "素鸭", "鸭腿", "豆腐", "铁板", "武昌鱼", "湘菜", "川菜", "粤菜", "鲁菜", "东北菜", "新疆菜",
        "西北菜", "云南菜", "本帮菜", "杭帮菜", "淮扬菜", "农家菜", "农家乐", "农家特色", "土菜", "西餐", "日料", "日本料理", "韩国料理", "泰国菜",
        "东南亚菜", "印度菜", "台湾菜", "素菜", "鱼馆", "冰室", "大盘鸡", "烤鱼", "烧烤店"
    ),
}
CATEGORY_PRIORITY = list(CATEGORY_KEYWORDS)
NAME_KEYWORD_WEIGHT = 5

print("已恢复为手工维护的餐饮关键词词典，不再读取搜狗外部词典。")
print(f"餐饮-正餐关键词数：{len(CATEGORY_KEYWORDS['餐饮-正餐'])}")


已恢复为手工维护的餐饮关键词词典，不再读取搜狗外部词典。
餐饮-正餐关键词数：124


## 第 3 步：读取餐馆基础表并统计名称关键词


In [51]:
with BASE_RESTAURANTS_PATH.open("r", encoding="utf-8-sig", newline="") as source:
    restaurants = list(csv.DictReader(source))

rest_index = {restaurant["restId"]: index for index, restaurant in enumerate(restaurants)}
name_hits = defaultdict(Counter)
for restaurant in restaurants:
    rest_id = restaurant["restId"]
    name = (restaurant.get("name") or "").strip()
    if not name:
        continue
    for category, keywords in CATEGORY_KEYWORDS.items():
        for keyword in keywords:
            if keyword.lower() in name.lower():
                name_hits[rest_id][category] += 1

print(f"餐馆记录数：{len(restaurants):,}")
print(f"名称中命中类别关键词的 restId 数：{len(name_hits):,}")


餐馆记录数：243,247
名称中命中类别关键词的 restId 数：101,531


## 第 4 步：流式扫描全部评论并统计类别关键词


In [52]:
from array import array

keyword_to_categories = defaultdict(set)
for category, keywords in CATEGORY_KEYWORDS.items():
    for keyword in keywords:
        keyword_to_categories[keyword.lower()].add(category)

all_keywords = sorted(keyword_to_categories, key=len, reverse=True)
keyword_pattern = re.compile("|".join(re.escape(keyword) for keyword in all_keywords), re.IGNORECASE)
comment_hit_arrays = {
    category: array("I", [0]) * len(restaurants)
    for category in CATEGORY_KEYWORDS
}
comment_rows = 0

with RATINGS_PATH.open("r", encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    for row in reader:
        comment = (row.get("comment") or "").strip()
        rest_id = row.get("restId")
        rest_position = rest_index.get(rest_id)
        if not comment or rest_position is None:
            continue
        comment_rows += 1
        matched_categories = set()
        for match in keyword_pattern.finditer(comment):
            matched_categories.update(keyword_to_categories[match.group(0).lower()])
        for category in matched_categories:
            comment_hit_arrays[category][rest_position] += 1

comment_hit_rest_ids = sum(
    any(comment_hit_arrays[category][index] for category in CATEGORY_KEYWORDS)
    for index in range(len(restaurants))
)
print(f"扫描的非空评论数：{comment_rows:,}")
print(f"评论中命中类别关键词的 restId 数：{comment_hit_rest_ids:,}")


扫描的非空评论数：4,107,409
评论中命中类别关键词的 restId 数：213,901


## 第 5 步：计算描述类别，并输出草稿表


In [53]:
def classify_restaurant(rest_id):
    position = rest_index[rest_id]
    scores = Counter({
        category: comment_hit_arrays[category][position]
        for category in CATEGORY_KEYWORDS
        if comment_hit_arrays[category][position]
    })
    for category, hit_count in name_hits.get(rest_id, {}).items():
        scores[category] += hit_count * NAME_KEYWORD_WEIGHT

    if not scores:
        return "不确定", "低", "名称和评论均未命中类别关键词"

    ranked = sorted(
        scores.items(),
        key=lambda item: (-item[1], CATEGORY_PRIORITY.index(item[0]))
    )
    top_category, top_score = ranked[0]
    second_score = ranked[1][1] if len(ranked) > 1 else 0
    name_evidence = name_hits.get(rest_id, {}).get(top_category, 0)
    comment_evidence = comment_hit_arrays[top_category][position]

    if name_evidence > 0 or (top_score >= 5 and top_score - second_score >= 2):
        confidence = "高"
    elif top_score >= 2 and top_score > second_score:
        confidence = "中"
    else:
        confidence = "低"

    evidence = f"名称命中{top_category}关键词{name_evidence}次；评论命中{comment_evidence}条；得分{top_score}"
    return top_category, confidence, evidence

category_counts = Counter()
for restaurant in restaurants:
    category, _, _ = classify_restaurant(restaurant["restId"])
    restaurant["描述类别"] = category
    category_counts[category] += 1

output_fields = list(restaurants[0].keys())
with OUTPUT_PATH.open("w", encoding="utf-8-sig", newline="") as target:
    writer = csv.DictWriter(target, fieldnames=output_fields)
    writer.writeheader()
    writer.writerows(restaurants)

summary_rows = [
    {"描述类别": category, "店家数量": count}
    for category, count in category_counts.most_common()
]
with SUMMARY_PATH.open("w", encoding="utf-8-sig", newline="") as target:
    writer = csv.DictWriter(target, fieldnames=["描述类别", "店家数量"])
    writer.writeheader()
    writer.writerows(summary_rows)

print(f"类别草稿表：{OUTPUT_PATH}")
print(f"类别汇总表：{SUMMARY_PATH}")
print(category_counts)


类别草稿表：D:\携程实习文件\项目2\推荐系统数据集\推荐系统数据集\店家\restaurants_category_draft.csv
类别汇总表：D:\携程实习文件\项目2\推荐系统数据集\推荐系统数据集\店家\restaurant_category_summary.csv
Counter({'餐饮-正餐': 124803, '餐饮-茶饮咖啡甜品': 30922, '餐饮-火锅烧烤': 23884, '不确定': 21162, '酒店/住宿': 9053, '购物零售': 8459, '景点/公园/公共场所': 5238, '生活服务': 5217, '休闲娱乐': 4645, '超市/便利店': 4353, '酒吧/夜生活': 3419, '教育培训/驾校': 1093, '电商平台/线上零售': 657, '市场/花鸟市场': 342})


## 第 6 步：查看样例


In [54]:
for restaurant in restaurants[:20]:
    print({
        "restId": restaurant["restId"],
        "name": restaurant.get("name"),
        "描述类别": restaurant["描述类别"],
    })


{'restId': '0', 'name': '', '描述类别': '购物零售'}
{'restId': '1', 'name': '', '描述类别': '不确定'}
{'restId': '2', 'name': '', '描述类别': '超市/便利店'}
{'restId': '3', 'name': '', '描述类别': '不确定'}
{'restId': '4', 'name': '', '描述类别': '景点/公园/公共场所'}
{'restId': '5', 'name': '', '描述类别': '景点/公园/公共场所'}
{'restId': '6', 'name': '', '描述类别': '不确定'}
{'restId': '7', 'name': '', '描述类别': '不确定'}
{'restId': '8', 'name': '', '描述类别': '不确定'}
{'restId': '9', 'name': '', '描述类别': '购物零售'}
{'restId': '10', 'name': '', '描述类别': '购物零售'}
{'restId': '11', 'name': '', '描述类别': '购物零售'}
{'restId': '12', 'name': '', '描述类别': '购物零售'}
{'restId': '13', 'name': '', '描述类别': '购物零售'}
{'restId': '14', 'name': '家乐福', '描述类别': '超市/便利店'}
{'restId': '15', 'name': '百佳', '描述类别': '超市/便利店'}
{'restId': '16', 'name': '', '描述类别': '超市/便利店'}
{'restId': '17', 'name': '好又多', '描述类别': '超市/便利店'}
{'restId': '18', 'name': '', '描述类别': '超市/便利店'}
{'restId': '19', 'name': '', '描述类别': '不确定'}


## 第 7 步：按类别随机抽取评论样本

默认抽取“不确定”类别的评论；将 `TARGET_CATEGORY` 改为任一类别名称，或设为 `None`，即可抽取指定类别或全部店家的随机评论。

In [55]:
import random

# RATING_SPLIT_DIR 已在首个代码单元格中定义。
TARGET_CATEGORY = "不确定"  # 例如："餐饮-正餐"；设为 None 表示不限制类别
SAMPLE_SIZE = 50
MIN_COMMENT_LENGTH = 8
RANDOM_SEED = 20260717

random.seed(RANDOM_SEED)
with OUTPUT_PATH.open("r", encoding="utf-8-sig", newline="") as source:
    final_restaurants = list(csv.DictReader(source))

category_by_rest_id = {row["restId"]: row["描述类别"] for row in final_restaurants}
name_by_rest_id = {row["restId"]: row.get("name", "") for row in final_restaurants}
samples = []
eligible_count = 0

for part_path in sorted(RATING_SPLIT_DIR.glob("ratings_part_*.csv")):
    with part_path.open("r", encoding="utf-8-sig", newline="") as source:
        for row in csv.DictReader(source):
            rest_id = row.get("restId")
            category = category_by_rest_id.get(rest_id)
            comment = (row.get("comment") or "").strip()
            if category is None or (TARGET_CATEGORY is not None and category != TARGET_CATEGORY):
                continue
            if len(comment) < MIN_COMMENT_LENGTH:
                continue

            eligible_count += 1
            record = {
                "restId": rest_id,
                "name": name_by_rest_id[rest_id],
                "描述类别": category,
                "comment": comment,
                "comment_length": len(comment),
            }
            if len(samples) < SAMPLE_SIZE:
                samples.append(record)
            else:
                replace_at = random.randrange(eligible_count)
                if replace_at < SAMPLE_SIZE:
                    samples[replace_at] = record

print(f"抽样范围：{TARGET_CATEGORY or '全部类别'}")
print(f"符合条件的评论数：{eligible_count:,}")
print(f"随机抽取的评论数：{len(samples):,}")
for sample in samples:
    print(sample)


抽样范围：不确定
符合条件的评论数：31,702
随机抽取的评论数：50
{'restId': '28584', 'name': '大老赵', '描述类别': '不确定', 'comment': '一家小店，以手擀面为主，位置不是很好找\n价格不贵，面比较劲道，因为我本身不怎么爱吃面，看了美娱来了陪朋友去吃了吃', 'comment_length': 56}
{'restId': '162603', 'name': '油炸鸡冠饺(熊家咀店)', '描述类别': '不确定', 'comment': '一个卖鸡冠饺和油香（糖糕）的小摊点。\n鸡冠饺和油香的个头都不算大，都是5毛一个。\n鸡冠饺里包的馅料主要是粉丝，味道还行。\n油香的皮是用糯米面和面粉混合而成的面做的，包的是白糖，热着吃，表皮是脆的，挺香的，又不是太甜，不错。', 'comment_length': 111}
{'restId': '195026', 'name': '', '描述类别': '不确定', 'comment': '设施比较好，环境也很好，票价也不高，但遗憾的是星期二那里不是半价\n比较喜欢小包的感觉，没有几个人，沙发也蛮舒服，每次都有躺下昏昏欲睡的冲动', 'comment_length': 69}
{'restId': '195222', 'name': '', '描述类别': '不确定', 'comment': '停车场有5-6层，每层几十个车位，所以去晚了要一圈一圈的绕上去的车位。\n停好车出来向左走就是二号线5号口。\n管理员大叔很NICE,可能每天来停车的人都熟了，看到我就问：你第一次来吧？\n感觉好亲切。', 'comment_length': 98}
{'restId': '91608', 'name': '', '描述类别': '不确定', 'comment': '有次和朋友闲逛来到这里，发现这里有好多家具卖！时尚的，古典的，有个性的，温馨的……应有尽有，跟某些家具城有的比！不过这里不可以拍照，不像宜家可以随便拍照，随便躺在床上试睡，随便坐在展出的凳子上休息。。。你一进去某个牌子的店内，店员就会形影不离地跟着你，感觉不太好……', 'comment_length': 133}
{'restId': '171231', 'name': '中华聚满楼', '描述类别': '不确

## 第 8 步：统计最终分类结果

直接读取最终的 `restaurants_category_draft.csv`，展示每个描述类别的店家数量和占比。

In [56]:
from collections import Counter

with OUTPUT_PATH.open("r", encoding="utf-8-sig", newline="") as source:
    final_rows = list(csv.DictReader(source))

final_category_counts = Counter(row["描述类别"] for row in final_rows)
total_restaurants = len(final_rows)
print(f"最终店家记录数：{total_restaurants:,}")
print(f"描述类别数：{len(final_category_counts)}")
print()
for category, count in final_category_counts.most_common():
    print(f"{category}：{count:,} 家，占比 {count / total_restaurants:.2%}")


最终店家记录数：243,247
描述类别数：14

餐饮-正餐：124,803 家，占比 51.31%
餐饮-茶饮咖啡甜品：30,922 家，占比 12.71%
餐饮-火锅烧烤：23,884 家，占比 9.82%
不确定：21,162 家，占比 8.70%
酒店/住宿：9,053 家，占比 3.72%
购物零售：8,459 家，占比 3.48%
景点/公园/公共场所：5,238 家，占比 2.15%
生活服务：5,217 家，占比 2.14%
休闲娱乐：4,645 家，占比 1.91%
超市/便利店：4,353 家，占比 1.79%
酒吧/夜生活：3,419 家，占比 1.41%
教育培训/驾校：1,093 家，占比 0.45%
电商平台/线上零售：657 家，占比 0.27%
市场/花鸟市场：342 家，占比 0.14%
